In [3]:
import time
import logging
from datetime import datetime, timedelta
from typing import Dict, List
from pathlib import Path

import requests
import yfinance as yf
import pandas as pd
import numpy as np
import json

# import config

from dotenv import load_dotenv
import os

load_dotenv()

True

In [4]:
# ── CONFIGURATION (insert here) ─────────────────────────────────────
TICKERS     = ['AAPL', 'MSFT', 'NVDA', 'GOOG']   # stock symbols
MACRO       = ['GC=F', 'CL=F']                    # Gold, Crude Oil futures
ALL_SYMBOLS = TICKERS + MACRO
PERIOD      = '5y'   # 1mo | 1y
INTERVAL    = '1mo'   # 1d | 1wk | 1mo | 1y

# Load API keys from .env 
GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')
NEWSAPI_KEY  = os.getenv('NEWSAPI_KEY',  '')
FRED_API_KEY = os.getenv('FRED_API_KEY', '')

print(f'Tickers : {TICKERS}')
print(f'Macro   : {MACRO}')
print(f'Period  : {PERIOD}')
print(f'Interval: {INTERVAL}')
print(f'Groq key: {"✅ set" if GROQ_API_KEY else "⚠️  not set"}')
print(f'News key: {"✅ set" if NEWSAPI_KEY  else "⚠️  not set"}')
print(f'FRED key: {"✅ set" if FRED_API_KEY  else "⚠️  not set"}')

Tickers : ['AAPL', 'MSFT', 'NVDA', 'GOOG']
Macro   : ['GC=F', 'CL=F']
Period  : 5y
Interval: 1mo
Groq key: ✅ set
News key: ✅ set
FRED key: ✅ set


In [5]:
# Output directories
for d in ['data/raw', 'data/processed', 'outputs/charts', 'outputs/reports']:
    Path(d).mkdir(parents=True, exist_ok=True)

In [6]:
def fetch_prices(symbols: List[str], period: str = '6mo', interval: str = '1d') -> Dict[str, pd.DataFrame]:
    results = {}
    for symbol in symbols:
        try:
            df = yf.Ticker(symbol).history(period=period, interval=interval, auto_adjust=True)
            if df.empty:
                print(f'  ⚠️  No data for {symbol}')
                continue
            df.index.name = 'Date'
            if hasattr(df.index, 'tz') and df.index.tz:
                df.index = df.index.tz_localize(None)
            safe = symbol.replace('=', '_')
            df.to_csv(f'data/raw/{safe}_prices.csv')
            results[symbol] = df
            print(f'  ✅ {symbol:10} {len(df):4d} rows  →  data/raw/{safe}_prices.csv')
        except Exception as e:
            print(f'  ❌ {symbol}: {e}')
        time.sleep(0.5)
    return results

print('MODULE 1 — Fetching price data …')
print('-' * 40)
raw_dfs = fetch_prices(ALL_SYMBOLS, period=PERIOD, interval=INTERVAL)
print(f'\n✅ {len(raw_dfs)} datasets saved to data/raw/')

MODULE 1 — Fetching price data …
----------------------------------------
  ✅ AAPL         60 rows  →  data/raw/AAPL_prices.csv
  ✅ MSFT         60 rows  →  data/raw/MSFT_prices.csv
  ✅ NVDA         60 rows  →  data/raw/NVDA_prices.csv
  ✅ GOOG         60 rows  →  data/raw/GOOG_prices.csv
  ✅ GC=F         51 rows  →  data/raw/GC_F_prices.csv
  ✅ CL=F         51 rows  →  data/raw/CL_F_prices.csv

✅ 6 datasets saved to data/raw/


In [7]:
def fetch_news(tickers: list, api_key: str, days_back: int = 30) -> dict:
    if not api_key:
        print('  ⚠️  NEWSAPI_KEY not set — skipping news')
        return {}

    # Auto-fetch company names from yfinance
    ticker_names = {}
    for t in tickers:
        try:
            info = yf.Ticker(t).info
            short_name = (info.get('shortName') or '').lower()
            first_word = short_name.split()[0] if short_name else ''
            ticker_names[t] = {t.lower(), first_word, short_name}
            ticker_names[t].discard('')
        except Exception:
            ticker_names[t] = {t.lower()}

    results = {}
    from_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
    for ticker in tickers:
        try:
            # Use company name in query for better coverage
            company_names = ticker_names.get(ticker, {ticker.lower()})
            first_word = next(
                (n for n in company_names if n != ticker.lower() and len(n) > 2),
                ticker
            )
            query = f'{ticker} OR "{first_word}" stock'

            resp = requests.get(
                'https://newsapi.org/v2/everything',
                params={
                    'q':        query,
                    'from':     from_date,
                    'language': 'en',
                    'sortBy':   'relevancy',
                    'pageSize': 100,
                    'apiKey':   api_key,
                },
                timeout=10
            )
            resp.raise_for_status()
            articles = resp.json().get('articles', [])

            filtered = []
            for a in articles:
                if not a.get('title'):
                    continue
                text = ((a.get('title') or '') + ' ' +
                        (a.get('description') or '')).lower()
                if any(name in text for name in company_names):
                    filtered.append({
                        'title':       a['title'].strip(),
                        'description': (a.get('description') or '').strip(),
                        'publishedAt': a.get('publishedAt', ''),
                        'source':      (a.get('source') or {}).get('name', ''),
                        'url':         a.get('url', ''),
                    })

            results[ticker] = filtered
            print(f'  ✅ {ticker:10} {len(filtered):3d} articles (filtered from {len(articles)})')
        except Exception as e:
            print(f'  ❌ {ticker} news: {e}')
            results[ticker] = []
        time.sleep(1)
    return results

print('MODULE 1 — Fetching news headlines …')
print('-' * 40)
news_data = fetch_news(TICKERS, api_key=NEWSAPI_KEY)

with open('data/raw/news_data.json', 'w', encoding='utf-8') as f:
    json.dump(news_data, f, ensure_ascii=False, indent=2)
print(f'\n✅ {sum(len(v) for v in news_data.values())} articles saved to data/raw/news_data.json')

MODULE 1 — Fetching news headlines …
----------------------------------------
  ✅ AAPL        28 articles (filtered from 89)
  ✅ MSFT        15 articles (filtered from 89)
  ✅ NVDA        12 articles (filtered from 89)
  ✅ GOOG         8 articles (filtered from 89)

✅ 63 articles saved to data/raw/news_data.json


In [8]:
config = {
    'TICKERS': TICKERS,
    'MACRO': MACRO,
    'ALL_SYMBOLS': ALL_SYMBOLS,
    'PERIOD': PERIOD,
    'INTERVAL': INTERVAL,
    'GROQ_API_KEY': GROQ_API_KEY,
    'NEWSAPI_KEY': NEWSAPI_KEY,
    'FRED_API_KEY': FRED_API_KEY,
}
with open('data/raw/finagent_config.json', 'w') as f:
    json.dump(config, f, indent=2)

In [9]:
first = TICKERS[0]
if first in raw_dfs:
    print(f'Sample: {first} (last 5 rows)')
    display(raw_dfs[first].tail())
    print(f'Date range: {raw_dfs[first].index[0].date()} → {raw_dfs[first].index[-1].date()}')
if news_data.get(first):
    print(f'\nSample news — {first}:')
    for a in news_data[first][:3]:
        print(f"  [{a['publishedAt'][:10]}] {a['title'][:80]}")

Sample: AAPL (last 5 rows)


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-01-01,271.755140,277.324780,242.968609,258.998840,1040017300,0.00,0.0
2026-02-01,259.547796,280.389081,254.976287,263.690094,988921000,0.26,0.0
2026-03-01,262.168457,266.284660,245.284004,253.556381,900035700,0.00,0.0
2026-04-01,253.846142,275.745964,245.473850,271.100250,907538500,0.00,0.0
2026-05-01,278.859985,311.399994,274.859985,308.820007,808170563,0.27,0.0


Date range: 2021-06-01 → 2026-05-01

Sample news — AAPL:
  [2026-05-03] Under Ternus, Apple Is Reportedly Entering a Spendy New Era
  [2026-05-06] The Weirdest Wearables From 100 Years Ago
  [2026-04-24] Best Apple Deals of the Week: Low Prices on Apple Watch, MacBook Air, AirTag 1, 
